In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Starting global inflation data processing...")

# ------------------------------------------------------------
# 1. Helper to load World Bank CSV
# ------------------------------------------------------------
def load_wb_csv(filepath, indicator_name):
    # World Bank CSV files have metadata in first 4 rows
    df = pd.read_csv(filepath, skiprows=4)

    # Keep only country info and years 1990-2023
    years = [str(y) for y in range(1990, 2024)]
    keep_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'] + years
    keep_cols = [col for col in keep_cols if col in df.columns]
    df = df[keep_cols]

    # Drop rows without a proper country code (aggregates, regions, etc.)
    df = df.dropna(subset=['Country Name', 'Country Code'])
    df = df[df['Country Code'].str.len() == 3]   # standard 3-letter codes

    # Melt years into one row per country-year
    melted = df.melt(
        id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
        value_vars=years,
        var_name='Year',
        value_name=indicator_name
    )

    melted['Year'] = pd.to_numeric(melted['Year'])
    melted = melted.dropna(subset=[indicator_name])
    melted = melted.rename(columns={'Country Name': 'Country', 'Country Code': 'economy'})

    print(f"  {indicator_name}: {len(melted)} rows loaded")
    return melted

# ------------------------------------------------------------
# 2. Load all indicator files
# ------------------------------------------------------------
print("\nLoading CSV files...")

# Update these paths if your files are located elsewhere
data_files = {
    'API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_4522': 'Inflation_Rate',
    'API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_3530': 'GDP_Growth',
    'API_NY.GDP.PCAP.KD_DS2_en_csv_v2_3781': 'GDP_Per_Capita',
    'API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_3359': 'Unemployment'
}

loaded_data = {}
for filename, indicator_name in data_files.items():
    filepath = f"{filename}.csv"
    if not os.path.exists(filepath):
        print(f"  File not found: {filepath}, skipping")
        continue
    try:
        df = load_wb_csv(filepath, indicator_name)
        loaded_data[indicator_name] = df
    except Exception as e:
        print(f"  Error loading {filepath}: {e}")

if len(loaded_data) < 1:
    print("No data loaded. Exiting.")
    exit()

# ------------------------------------------------------------
# 3. Merge all indicators
# ------------------------------------------------------------
print("\nMerging indicators...")

# Start with inflation data
merged = loaded_data['Inflation_Rate'].copy()

for indicator in ['GDP_Growth', 'GDP_Per_Capita', 'Unemployment']:
    if indicator in loaded_data:
        merged = merged.merge(
            loaded_data[indicator][['economy', 'Country', 'Year', indicator]],
            on=['economy', 'Country', 'Year'],
            how='left'
        )

print(f"  Merged dataset shape: {merged.shape}")

# ------------------------------------------------------------
# 4. Add region and income level
# ------------------------------------------------------------
print("\nAdding region and income information...")

region_mapping = {
    'USA': 'North America', 'CAN': 'North America', 'MEX': 'North America',
    'BRA': 'South America', 'ARG': 'South America', 'CHL': 'South America',
    'COL': 'South America', 'PER': 'South America', 'VEN': 'South America',
    'ECU': 'South America', 'BOL': 'South America', 'PRY': 'South America',
    'URY': 'South America',
    'GBR': 'Europe', 'DEU': 'Europe', 'FRA': 'Europe', 'ITA': 'Europe',
    'ESP': 'Europe', 'NLD': 'Europe', 'BEL': 'Europe', 'CHE': 'Europe',
    'SWE': 'Europe', 'NOR': 'Europe', 'DNK': 'Europe', 'FIN': 'Europe',
    'POL': 'Europe', 'TUR': 'Europe', 'RUS': 'Europe', 'PRT': 'Europe',
    'AUT': 'Europe', 'GRC': 'Europe', 'IRL': 'Europe', 'ISR': 'Europe',
    'CHN': 'Asia', 'JPN': 'Asia', 'KOR': 'Asia', 'IND': 'Asia',
    'IDN': 'Asia', 'SAU': 'Asia', 'IRN': 'Asia', 'PAK': 'Asia',
    'BGD': 'Asia', 'PHL': 'Asia', 'VNM': 'Asia', 'THA': 'Asia',
    'MYS': 'Asia', 'SGP': 'Asia', 'LKA': 'Asia', 'NPL': 'Asia',
    'ZAF': 'Africa', 'NGA': 'Africa', 'EGY': 'Africa', 'KEN': 'Africa',
    'ETH': 'Africa', 'GHA': 'Africa', 'MAR': 'Africa', 'TUN': 'Africa',
    'DZA': 'Africa', 'AGO': 'Africa', 'CMR': 'Africa', 'CIV': 'Africa',
    'AUS': 'Oceania', 'NZL': 'Oceania',
    'ARE': 'Middle East', 'QAT': 'Middle East', 'KWT': 'Middle East',
    'BHR': 'Middle East', 'OMN': 'Middle East', 'IRQ': 'Middle East',
}

income_mapping = {
    'USA': 'High', 'CAN': 'High', 'GBR': 'High', 'DEU': 'High', 'FRA': 'High',
    'ITA': 'High', 'ESP': 'High', 'NLD': 'High', 'BEL': 'High', 'CHE': 'High',
    'SWE': 'High', 'NOR': 'High', 'DNK': 'High', 'FIN': 'High', 'AUS': 'High',
    'NZL': 'High', 'JPN': 'High', 'KOR': 'High', 'SAU': 'High', 'SGP': 'High',
    'ISR': 'High', 'ARE': 'High', 'QAT': 'High', 'KWT': 'High', 'BHR': 'High',
    'OMN': 'High', 'IRL': 'High', 'PRT': 'High', 'GRC': 'High', 'AUT': 'High',
    'CHL': 'High', 'URY': 'High',
    'MEX': 'Upper Middle', 'BRA': 'Upper Middle', 'ARG': 'Upper Middle',
    'CHN': 'Upper Middle', 'TUR': 'Upper Middle', 'RUS': 'Upper Middle',
    'ZAF': 'Upper Middle', 'MYS': 'Upper Middle', 'THA': 'Upper Middle',
    'COL': 'Upper Middle', 'PER': 'Upper Middle', 'ECU': 'Upper Middle',
    'BOL': 'Upper Middle', 'PRY': 'Upper Middle', 'DZA': 'Upper Middle',
    'TUN': 'Upper Middle',
    'IND': 'Lower Middle', 'IDN': 'Lower Middle', 'PHL': 'Lower Middle',
    'VNM': 'Lower Middle', 'MAR': 'Lower Middle', 'EGY': 'Lower Middle',
    'NGA': 'Lower Middle', 'KEN': 'Lower Middle', 'PAK': 'Lower Middle',
    'BGD': 'Lower Middle', 'IRN': 'Lower Middle', 'LKA': 'Lower Middle',
    'NPL': 'Lower Middle', 'CMR': 'Lower Middle', 'CIV': 'Lower Middle',
    'AGO': 'Lower Middle',
    'ETH': 'Low', 'MWI': 'Low', 'UGA': 'Low', 'MOZ': 'Low',
    'TZA': 'Low', 'RWA': 'Low',
}

merged['Region'] = merged['economy'].map(region_mapping).fillna('Other')
merged['IncomeLevel'] = merged['economy'].map(income_mapping).fillna('Unknown')

# ------------------------------------------------------------
# 5. Add derived fields
# ------------------------------------------------------------
print("Adding derived fields...")

merged['Decade'] = (merged['Year'] // 10) * 10

def inflation_category(rate):
    if pd.isna(rate):
        return 'Unknown'
    if rate < 0:
        return 'Deflation'
    if rate < 2:
        return 'Low'
    if rate < 5:
        return 'Moderate'
    if rate < 10:
        return 'High'
    return 'Very High'

merged['Inflation_Category'] = merged['Inflation_Rate'].apply(inflation_category)
merged['High_Inflation'] = merged['Inflation_Rate'] > 10

# ------------------------------------------------------------
# 6. Save cleaned data
# ------------------------------------------------------------
output_file = 'inflation_clean.csv'
merged.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

# ------------------------------------------------------------
# 7. Print summary
# ------------------------------------------------------------
print("\nDataset summary:")
print(f"  Countries: {merged['Country'].nunique()}")
print(f"  Years: {merged['Year'].min()} - {merged['Year'].max()}")
print(f"  Observations: {len(merged)}")
print(f"  Inflation range: {merged['Inflation_Rate'].min():.1f}% to {merged['Inflation_Rate'].max():.1f}%")

print("\nTop 10 average inflation (1990-2023):")
top_inf = merged.groupby('Country')['Inflation_Rate'].mean().sort_values(ascending=False).head(10)
for country, rate in top_inf.items():
    print(f"  {country}: {rate:.1f}%")

print("\nData completeness:")
for col in ['Inflation_Rate', 'GDP_Growth', 'GDP_Per_Capita', 'Unemployment']:
    if col in merged.columns:
        n = merged[col].notna().sum()
        pct = (n / len(merged)) * 100
        print(f"  {col}: {n}/{len(merged)} ({pct:.1f}%)")

print("\nRegion counts:")
print(merged['Region'].value_counts().to_string())

print("Processing complete.")